<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Tool Dispatcher Pattern </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Tool Dispatcher Pattern

**Purpose:** This notebook demonstrates the **Tool Dispatcher Architecture** — how tools are registered, dispatched, executed, and how errors are handled in an AI agent system.

**Key Concepts Covered:**
- Tool Registry Pattern
- Tool Dispatcher
- Handling tool results
- Basic error handling in tool execution

Building a Travel Planning Agent for **Jeddah, Saudi Arabia**.  
The agent helps users with **real live weather**, tourist places, and trip cost estimation.


## 1. Setup

In [ ]:
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass
import requests

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
Setup complete


In [ ]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

## 2. Tool Definitions

We define three practical tools for planning a trip to Jeddah, Saudi Arabia.

- `get_weather`: Fetches **real live weather** data from Open-Meteo API.
- `get_tourist_places`: Scrapes real tourism information.
- `calculate_trip_cost`: Calculates estimated trip cost.

In [ ]:
from langchain_core.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def get_weather(city: str) -> str:
    """Get real live weather for Jeddah using Open-Meteo API."""
    try:
        lat, lon = 21.4858, 39.1925
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,wind_speed_10m"
        response = requests.get(url, timeout=10)
        data = response.json()

        current = data["current"]
        return f"Current weather in Jeddah, Saudi Arabia: {current['temperature_2m']}°C, Humidity: {current['relative_humidity_2m']}%, Wind: {current['wind_speed_10m']} km/h."
    except Exception as e:
        return f"Failed to fetch weather. Error: {str(e)}"

@tool
def get_tourist_places(city: str) -> str:
    """Get popular tourist places in Jeddah with clean and readable output."""
    try:
        places = """Popular Tourist Attractions in Jeddah, Saudi Arabia:

1. King Fahd Fountain (world's tallest fountain)
2. Al-Balad Historic District (UNESCO World Heritage site)
3. Jeddah Corniche (beautiful waterfront promenade)
4. Masjid Al-Rahma (Floating Mosque)
5. Red Sea Mall (major shopping destination)
6. King Abdullah Economic City"""

        overview = "Jeddah is the commercial capital of Saudi Arabia and a major gateway for pilgrims heading to Mecca. It is known for its mix of modern infrastructure and rich historic culture."

        return places + "\n\nOverview:\n" + overview
    except Exception as e:
        return f"Could not fetch tourist information. Error: {str(e)}"

@tool
def calculate_trip_cost(days: int, people: int, budget_level: str = "medium") -> str:
    """Calculate estimated trip cost for Jeddah in SAR and USD."""
    base_hotel = 450 if budget_level == "luxury" else 200 if budget_level == "medium" else 100
    food_per_day = 70 if budget_level == "luxury" else 40 if budget_level == "medium" else 25
    transport = 150

    total_sar = (base_hotel * days * people) + (food_per_day * days * people) + (transport * days)
    total_usd = round(total_sar / 3.75)

    return f"""Estimated cost for {days} days and {people} people in Jeddah ({budget_level} budget):

SAR {total_sar:,}
≈ USD {total_usd:,}"""

## 3. Tool Registry Pattern

The Tool Registry acts as a central directory of all available tools.

In [ ]:
tool_registry = {
    "get_weather": get_weather,
    "get_tourist_places": get_tourist_places,
    "calculate_trip_cost": calculate_trip_cost
}

print(f"Tool Registry initialized with {len(tool_registry)} tools.")

Tool Registry initialized with 3 tools.


## 4. Tool Dispatcher

The Tool Dispatcher is responsible for safely executing the requested tool and handling errors.

In [ ]:
def tool_dispatcher(tool_name: str, **kwargs):
    """Dispatches the tool call and returns standardized result."""
    if tool_name not in tool_registry:
        return {"status": "error", "tool": tool_name, "error": f"Tool '{tool_name}' not found."}

    try:
        tool = tool_registry[tool_name]
        result = tool.run(kwargs) if hasattr(tool, 'run') else tool(**kwargs)
        return {"status": "success", "tool": tool_name, "result": result}
    except Exception as e:
        return {"status": "error", "tool": tool_name, "error": str(e)}

## 5. Testing with Real Data for Jeddah

In [ ]:
print("1. Real Weather in Jeddah:")
print(tool_dispatcher("get_weather", city="Jeddah"))

print("\n2. Tourist Places in Jeddah:")
print(tool_dispatcher("get_tourist_places", city="Jeddah"))

print("\n3. Trip Cost Calculation:")
print(tool_dispatcher("calculate_trip_cost", days=5, people=2, budget_level="medium"))

1. Real Weather in Jeddah:
{'status': 'success', 'tool': 'get_weather', 'result': 'Real weather in Jeddah, Saudi Arabia: 29.1°C, Humidity: 57%, Wind: 16.8 km/h.'}

2. Tourist Places in Jeddah:
{'status': 'success', 'tool': 'get_tourist_places', 'result': "Popular places in Jeddah, Saudi Arabia:\n1. King Fahd's Fountain\n2. Al-Balad Historic District\n3. Jeddah Corniche\n4. Floating Mosque (Masjid Al-Rahma)\n5. Red Sea Mall\n\n\n\nJeddah[a]lies on theRed Seacoast in theHijazregion ofSaudi Arabia. It serves as agovernorate, is the largest city inMecca Province, and, after the capital,Riyadh, ranks as the country'ssecond-largest city. Jeddah is the commercial center of the country. It is not known when Jeddah was founded, but Jeddah's prominence grew in 647 when theCaliphUthmanmade it a travel hub servingMuslimtravelers going to the holy city ofMeccaforIslamic pilgrimage. Since those times, Jeddah has served as a gateway for millions of pilgrims who have arrived in Saudi Arabia.[4][5]\n\n